# EUR/USD Pipeline

This notebook is the wiring entrypoint for the project. It runs the pipeline in `dry_run` mode by default.

In [1]:
from pathlib import Path
import sys


def add_src_to_path() -> None:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data").exists():
            sys.path.insert(0, str(p / "src"))
            return


add_src_to_path()

from eur_usd.pipeline.run import run_pipeline

result = run_pipeline(dry_run=True, verbose=True)
result

=== EUR/USD Pipeline (starter) ===
Dirty ticks: C:\Users\user\EUR_USD_Project\data\dirty\ticks\raw_source
Per-second output: C:\Users\user\EUR_USD_Project\data\interim\per_second
Events output: C:\Users\user\EUR_USD_Project\data\processed\events
Mode: dry-run


{'dry_run': True,
 'ticks_to_per_second': {'dry_run': True,
  'input_dir_exists': True,
  'output_dir_exists': True},
 'per_second_to_events': {'dry_run': True,
  'input_dir_exists': True,
  'output_dir_exists': True}}

## US Calendar (2020–2026)

Generates a date-level calendar with weekend, US holiday, and expected-data flags.  
Saved to `data/dirty/calendar/`.

In [2]:
import pandas as pd
import holidays

START = "2020-01-01"
END = "2026-12-31"

dates = pd.date_range(START, END, freq="D")
us_holidays = holidays.US(years=range(2020, 2027))

cal = pd.DataFrame({"date": dates})
cal["year"] = cal["date"].dt.year
cal["month"] = cal["date"].dt.month
cal["quarter"] = cal["date"].dt.quarter
cal["day_of_week"] = cal["date"].dt.dayofweek
cal["day_name"] = cal["date"].dt.day_name()
cal["is_weekend"] = cal["day_of_week"].isin([5, 6])

holiday_dates = set(pd.Timestamp(d) for d in us_holidays.keys())
cal["is_us_holiday"] = cal["date"].isin(holiday_dates)
cal["us_holiday_name"] = cal["date"].map(lambda d: us_holidays.get(d.date(), ""))
cal["is_expected_data"] = ~cal["is_weekend"]

print(f"Calendar: {len(cal)} days  ({cal['date'].min().date()} -> {cal['date'].max().date()})")
print(f"Weekends: {cal['is_weekend'].sum()}")
print(f"US holidays: {cal['is_us_holiday'].sum()}")
print(f"Expected-data days: {cal['is_expected_data'].sum()}")
cal.head(10)

Calendar: 2557 days  (2020-01-01 -> 2026-12-31)
Weekends: 730
US holidays: 86
Expected-data days: 1827


,date,year,month,quarter,day_of_week,day_name,is_weekend,is_us_holiday,us_holiday_name,is_expected_data
0,2020-01-01,2020,1,1,2,Wednesday,False,True,New Year's Day,True
1,2020-01-02,2020,1,1,3,Thursday,False,False,,True
2,2020-01-03,2020,1,1,4,Friday,False,False,,True
3,2020-01-04,2020,1,1,5,Saturday,True,False,,False
4,2020-01-05,2020,1,1,6,Sunday,True,False,,False
5,2020-01-06,2020,1,1,0,Monday,False,False,,True
6,2020-01-07,2020,1,1,1,Tuesday,False,False,,True
7,2020-01-08,2020,1,1,2,Wednesday,False,False,,True
8,2020-01-09,2020,1,1,3,Thursday,False,False,,True
9,2020-01-10,2020,1,1,4,Friday,False,False,,True


In [3]:
spot_checks = {
    "2020-01-01": "New Year's Day",
    "2021-07-05": "Independence Day (Observed)",
    "2023-01-02": "New Year's Day (Observed)",
    "2024-12-25": "Christmas Day",
    "2025-11-27": "Thanksgiving",
}

for date_str, expected_name in spot_checks.items():
    row = cal.loc[cal["date"] == date_str]
    assert len(row) == 1, f"Missing date: {date_str}"
    assert row["is_us_holiday"].iloc[0], f"{date_str} not flagged as holiday"
    actual = row["us_holiday_name"].iloc[0]
    print(f"  {date_str}: {actual}")

assert len(cal) == len(cal["date"].unique()), "Duplicate dates found"
assert cal["date"].is_monotonic_increasing, "Dates not sorted"
assert cal.loc[cal["is_weekend"], "day_of_week"].isin([5, 6]).all(), "Weekend flag mismatch"

print("\nAll spot-checks and integrity checks passed.")

  2020-01-01: New Year's Day
  2021-07-05: Independence Day (observed)
  2023-01-02: New Year's Day (observed)
  2024-12-25: Christmas Day
  2025-11-27: Thanksgiving Day

All spot-checks and integrity checks passed.


In [4]:
cal_dir = Path("data/dirty/calendar")
cal_dir.mkdir(parents=True, exist_ok=True)

parquet_path = cal_dir / "us_calendar.parquet"
csv_path = cal_dir / "us_calendar.csv"

cal.to_parquet(parquet_path, index=False)
cal.to_csv(csv_path, index=False)

print(f"Saved: {parquet_path}  ({parquet_path.stat().st_size / 1024:.1f} KB)")
print(f"Saved: {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")

Saved: data\dirty\calendar\us_calendar.parquet  (28.2 KB)
Saved: data\dirty\calendar\us_calendar.csv  (124.9 KB)


In [5]:
cal.loc[cal["is_us_holiday"], ["date", "day_name", "us_holiday_name"]]

,date,day_name,us_holiday_name
0,2020-01-01,Wednesday,New Year's Day
19,2020-01-20,Monday,Martin Luther King Jr. Day
47,2020-02-17,Monday,Washington's Birthday
145,2020-05-25,Monday,Memorial Day
184,2020-07-03,Friday,Independence Day (observed)
...,...,...,...
2441,2026-09-07,Monday,Labor Day
2476,2026-10-12,Monday,Columbus Day
2506,2026-11-11,Wednesday,Veterans Day
2521,2026-11-26,Thursday,Thanksgiving Day
